In [6]:
import numpy as np
import matplotlib.pyplot as plt
import json
import pandas as pd

In [21]:
def two_panel_entity_tag_figure(
    df,
    label_col="label",
    figsize=(14, 8),
    min_display_frac=0.01,
    panel_a_as_percent=True,
    sort_desc=True,
    save_path=None
):
    """
    Create a two-panel figure:
      Panel A: overall proportion of each entity type
      Panel B: 100% stacked bar chart of semantic-tag composition within each entity type

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe where:
        - label_col = entity type
        - all other columns = semantic tag counts
    label_col : str
        Column containing entity type labels
    figsize : tuple
        Figure size
    min_display_frac : float
        Minimum within-row fraction to annotate in Panel B
    panel_a_as_percent : bool
        If True, Panel A x-axis is shown as percentages
    sort_desc : bool
        If True, sort entity types by total count descending
    save_path : str or None
        If provided, save figure to this path
    """

    # ---- prepare data ----
    cols = [c for c in df.columns if c != label_col]
    plot_df = df.copy()

    # total count per entity type
    plot_df["total"] = plot_df[cols].sum(axis=1)

    # remove rows with zero total
    plot_df = plot_df.loc[plot_df["total"] > 0].copy()

    # sort entity types by total count
    if sort_desc:
        plot_df = plot_df.sort_values("total", ascending=False).reset_index(drop=True)

    entity_labels = plot_df[label_col].tolist()
    entity_totals = plot_df["total"].to_numpy()
    grand_total = entity_totals.sum()
    entity_props = entity_totals / grand_total

    # proportions within each entity type for panel B
    prop_df = plot_df[cols].div(plot_df["total"], axis=0)

    # color map for semantic tags
    cmap = plt.get_cmap("tab20")
    col_to_color = {c: cmap(i % cmap.N) for i, c in enumerate(cols)}

    # ---- figure layout ----
    fig, (ax1, ax2) = plt.subplots(
        ncols=2,
        figsize=figsize,
        gridspec_kw={"width_ratios": [1.0, 1.8]}
    )

    # =========================
    # Panel A: entity type distribution
    # =========================
    y = np.arange(len(entity_labels))

    if panel_a_as_percent:
        panel_a_vals = entity_props * 100
        xlabel_a = "Proportion of all entities (%)"
    else:
        panel_a_vals = entity_totals
        xlabel_a = "Number of entities"

    bars = ax1.barh(y, panel_a_vals)

    ax1.set_yticks(y)
    ax1.set_yticklabels(entity_labels)
    ax1.invert_yaxis()  # top = largest
    ax1.set_xlabel(xlabel_a)
    ax1.set_title("A. Overall distribution of entity domains")

    # annotate count + percent
    for i, (bar, count, prop) in enumerate(zip(bars, entity_totals, entity_props)):
        x = bar.get_width()
        txt = f"n={count:,} ({prop*100:.2f}%)"
        pad = max(panel_a_vals) * 0.01
        ax1.text(x + pad, bar.get_y() + bar.get_height()/2, txt,
                 va="center", fontsize=9)

    # give a bit of room on the right for labels
    ax1.set_xlim(0, max(panel_a_vals) * 1.35)

    # =========================
    # Panel B: 100% stacked composition
    # =========================
    left = np.zeros(len(plot_df))

    for c in cols:
        vals = prop_df[c].to_numpy()
        bars = ax2.barh(
            y,
            vals,
            left=left,
            label=c,
        )

        # annotate only sufficiently large segments
        for i, (v, lft) in enumerate(zip(vals, left)):
            if v >= min_display_frac:
                ax2.text(
                    lft + v / 2,
                    i,
                    f"{v*100:.2f}%",
                    ha="center",
                    va="center",
                    fontsize=8
                )
        left += vals

    ax2.set_yticks(y)
    ax2.set_yticklabels(entity_labels)
    ax2.invert_yaxis()
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("Semantic-tag composition within entity domain")
    ax2.set_title("B. Semantic-tag composition within each entity domain")

    # x-axis as percentages
    xticks = np.linspace(0, 1, 6)
    ax2.set_xticks(xticks)
    ax2.set_xticklabels([f"{int(x*100)}%" for x in xticks])

    # legend
    handles = [
        plt.Line2D([0], [0], marker='s', linestyle='',
                   markerfacecolor=col_to_color[c], markeredgecolor=col_to_color[c],
                   markersize=10, label=c)
        for c in cols
    ]
    # recolor plotted bars to match legend mapping
    for container, c in zip(ax2.containers, cols):
        for patch in container:
            patch.set_facecolor(col_to_color[c])
            patch.set_edgecolor("white")
            patch.set_linewidth(0.8)

    ax2.legend(
        handles=handles,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        title="Semantic tag"
    )

    # style tweaks
    for ax in (ax1, ax2):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
import json
import os
import pandas as pd

dataset_path = "YOUR_PATH_TO_DATASET.json"  # Update this to your actual dataset path

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset file not found: {dataset_path}")

if os.path.getsize(dataset_path) == 0:
    raise ValueError(f"Dataset file is empty: {dataset_path}")

with open(dataset_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

if not isinstance(data, list):
    raise TypeError("Expected a top-level JSON list of records.")

rows = []
for record in data:
    base = {
        'id': record.get('id'),
        'text': record.get('text'),
        'pdx': record.get('pdx'),
        'clinic': record.get('clinic'),
        'type': record.get('type'),
    }

    entities = record.get('entity', [])
    if not isinstance(entities, list):
        entities = [entities]

    for entity in entities:
        if not isinstance(entity, dict):
            continue

        row = dict(base)
        row.update({
            'start_index': entity.get('start_index'),
            'end_index': entity.get('end_index'),
            'label': entity.get('label'),
            'semantic_tag': entity.get('label'),
            'substring': entity.get('substring'),
            'Target code': entity.get('Target code'),
            'Target display': entity.get('Target display'),
            'Relationship type code': entity.get('Relationship type code'),
            'No map flag': entity.get('No map flag'),
            'Status': entity.get('Status'),
        })
        rows.append(row)

df = pd.DataFrame(rows)

In [ ]:
# Number of rows (height of the DF)
print(f"Number of rows in df: {len(df)}")

# Number of rows per semantic_tag
print("\nNumber of concepts per semantic_tag:")
semantic_tag_counts = df['semantic_tag'].value_counts()
semantic_tag_percentages = (semantic_tag_counts / len(df)) * 100
semantic_tag_df = pd.DataFrame({'count': semantic_tag_counts, 'percentage': semantic_tag_percentages.round(2)})
display(semantic_tag_df)

# Number of rows per label
print("\nNumber of concepts per domain:")
label_counts = df['label'].value_counts()
label_percentages = (label_counts / len(df)) * 100
label_df = pd.DataFrame({'count': label_counts, 'percentage': label_percentages.round(2)})
display(label_df)

# 10 most frequent target display
print("\nTop 10 most frequent concepts:")
target_display_counts = df['Target display'].value_counts().head(10)
target_display_percentages = (target_display_counts / len(df)) * 100
target_display_df = pd.DataFrame({'count': target_display_counts, 'percentage': target_display_percentages.round(2)})
display(target_display_df)


# Number of rows per one id reported as median(Percentile25th-Percentile75th)
rows_per_id = df['id'].value_counts()
median_rows = rows_per_id.median()
percentile_25 = rows_per_id.quantile(0.25)
percentile_75 = rows_per_id.quantile(0.75)

print(f"\nNumber of rows per id (median(25th percentile-75th percentile)): {median_rows} ({percentile_25}-{percentile_75})")

In [ ]:
semantic_tag_per_label = (
    df.groupby('label')['semantic_tag']
      .value_counts()
      .unstack(fill_value=0)
      .reset_index()
)
two_panel_entity_tag_figure(semantic_tag_per_label, label_col="label")